# Лабораторная работа № 7. Выбор модели, кросс-валидация и интерпретация

**Курс:** Классическое машинное обучение, 4 курс прикладной математики

## Цель работы

Освоить методы кросс-валидации, GridSearch, построение пайплайнов и интерпретацию моделей.

**Используемые инструменты:** `sklearn.pipeline`, `sklearn.model_selection`, `shap`, `matplotlib`.

### Регламент сдачи

Работа сдаётся в виде этого же ноутбука, дополненного вашим кодом. Обязательно:

1. Читаемый код с комментариями.
2. Визуализации (графики, таблицы).
3. **Текстовый вывод после каждого задания** — не только код, но и объяснение результата.
4. Финальный вывод по работе.

**Критерии оценки:** корректность реализации — 30 %, качество визуализаций и анализа — 20 %,
обоснованность выводов — 20 %, сравнение с эталонными реализациями — 15 %,
оригинальность и дополнительная работа — 15 %.

> Ячейки, помеченные `# TODO`, нужно заполнить самостоятельно.
> Ячейки с готовым кодом можно просто выполнить — они подготавливают данные и графики.

## Подготовка окружения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
sns.set_palette("viridis")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.3, stratify=data.target, random_state=RANDOM_STATE)

feature_names = data.feature_names
print("train:", X_train.shape)

## Задание 1. Пайплайн

Соберите цепочку

$$\text{StandardScaler} \rightarrow \text{PCA}(n\_components=10) \rightarrow \text{SVC}(kernel='rbf')$$

**Зачем пайплайн?** При кросс-валидации масштабирование и PCA должны обучаться
**только на обучающих фолдах**. Если сделать `scaler.fit(X)` до разбиения, информация
о тестовых объектах просочится в модель, и оценка качества будет завышенной.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

# TODO: соберите Pipeline из трёх шагов и обучите его как обычную модель

## Задание 2. GridSearchCV

Переберите сетку:

* $C \in \{0.1,\ 1,\ 10,\ 100\}$,
* $\gamma \in \{0.01,\ 0.1,\ 1,\ 10\}$,
* число компонент PCA $k \in \{5,\ 10,\ 15,\ 20\}$.

Обратите внимание на синтаксис имён параметров: `имя_шага__параметр`, например `svc__C`.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    # TODO: заполните сетку
}

# TODO: GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
#       Выведите best_params_ и best_score_. Сколько всего обучений выполнилось?

## Задание 3. GridSearch против RandomizedSearch

Запустите `RandomizedSearchCV` с 20 итерациями по той же (или более широкой непрерывной)
сетке. Сравните по времени работы и по качеству.

In [ ]:
import time
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# TODO: замерьте время обоих поисков через time.perf_counter(),
#       сравните best_score_ и число обученных моделей.
#       Сделайте вывод: когда случайный поиск предпочтительнее полного перебора?

## Задание 4. Финальная модель

Обучите лучшую модель на **всей** обучающей выборке и оцените один раз на тестовой.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay

# TODO: оцените финальную модель. Сравните оценку на кросс-валидации с оценкой на тесте —
#       насколько они близки? Если CV-оценка заметно выше, что это значит?

## Задание 5. Интерпретация через SHAP

SHAP приписывает каждому признаку вклад в конкретное предсказание, опираясь на значения Шепли
из теории кооперативных игр: вклад признака усредняется по всем возможным «коалициям» признаков.

> Если `shap` и `xgboost` не установлены: `pip install shap xgboost`.
> Как запасной вариант — `sklearn.inspection.permutation_importance` и `PartialDependenceDisplay`.

In [ ]:
# TODO: обучите XGBClassifier (или GradientBoostingClassifier) на тех же данных

In [ ]:
# TODO: explainer = shap.TreeExplainer(model); shap_values = explainer(X_test)
#       Постройте:
#         - shap.summary_plot(shap_values, X_test, feature_names=feature_names)
#         - shap.plots.waterfall(shap_values[0])  — разбор одного объекта
#       Какие 5 признаков влияют сильнее всего? Совпадают ли они с feature_importances_?

**Вывод:** *чем SHAP-важность отличается от `feature_importances_` дерева?
Почему первая может быть надёжнее?*

## Дополнительное задание. Вложенная кросс-валидация

Обычная схема «подобрали гиперпараметры по CV → взяли best_score_ как оценку качества»
даёт **смещённую вверх** оценку: гиперпараметры подогнаны под те же фолды.

Nested CV разделяет задачи: внешний цикл оценивает качество, внутренний — подбирает параметры.

```python
inner = KFold(n_splits=3, shuffle=True, random_state=0)
outer = KFold(n_splits=5, shuffle=True, random_state=0)
clf = GridSearchCV(pipe, param_grid, cv=inner, scoring="roc_auc")
scores = cross_val_score(clf, X, y, cv=outer, scoring="roc_auc")
```

In [ ]:
# TODO: реализуйте вложенную кросс-валидацию и сравните её оценку
#       с best_score_ из обычного GridSearchCV. Насколько велико смещение?

## Финальный вывод

*Напишите здесь связный вывод по работе (5–10 предложений):*

- какие методы вы применили и почему;
- какие результаты получили в числах;
- где реализация «с нуля» разошлась с эталоном из `sklearn` и в чём причина;
- что бы вы улучшили, будь у вас больше времени.